# Digit Recognizer — Tracking & Hyperparameter Tuning

This notebook re-implements the digit classifier with extensive metric tracking
so that every interesting quantity (loss, accuracy, parameter weights, gradients,
learning rate) is logged per step / per epoch and can be visualised afterwards.

Hyperparameter tuning is performed with **Optuna** in combination with
**K-Fold cross-validation (cv = 5)**. All PyTorch building blocks follow the
official documentation (`torch.optim`, `torch.nn`, `torch.utils.data`,
`torch.amp`, etc.).


## 1. Imports

In [ ]:
import math
import time
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import optuna
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset, Subset

import optuna.visualization.matplotlib as ovm

sns.set_theme(context="notebook", style="whitegrid")


## 2. Configuration

In [ ]:
SEED = 99
RESOLUTION = 32
N_SPLITS = 5            # K-Fold cv
N_OPTUNA_TRIALS = 15    # Optuna trials
TUNE_EPOCHS = 4         # epochs per fold during tuning (kept short)
FINAL_EPOCHS = 20       # epochs for the final retrain on best params
BATCH_SIZE_FALLBACK = 256

torch.manual_seed(SEED)
np.random.seed(SEED)

device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")


## 3. Dataset

In [ ]:
import kagglehub

temp_dir = Path.cwd() / ".temp"
if temp_dir.parent.name != "model":
    raise RuntimeError("Run this notebook from the `model/` directory.")
temp_dir.mkdir(exist_ok=True)

dataset_path = (
    Path(
        kagglehub.dataset_download(
            "metricasecuador/handwritten-digits-version-1-hwd-v1",
            output_dir=str(temp_dir / "dataset"),
        )
    )
    / "HWD-V1"
)
dataset_path


In [ ]:
def dataset_all_split(dataset, split, random_state=None):
    train, test = {}, {}
    rng = np.random.default_rng(random_state)
    for digit, items in dataset.items():
        shuffled = list(items)
        rng.shuffle(shuffled)
        idx = int(len(shuffled) * split)
        train[digit] = shuffled[:idx]
        test[digit] = shuffled[idx:]
    return train, test


def dataset_dict_to_list(d):
    return [(item, digit) for digit, items in d.items() for item in items]


DATASET_ITEMS_ALL = {
    digit: list(dataset_path.glob(f"HWD-V1-Standard/{digit}/*.png"))
    for digit in range(10)
}
DATASET_ITEMS_TRAIN, DATASET_ITEMS_TEST = dataset_all_split(
    DATASET_ITEMS_ALL, 0.7, random_state=SEED
)
DATASET_ITEMS_TRAIN_LIST = dataset_dict_to_list(DATASET_ITEMS_TRAIN)
DATASET_ITEMS_TEST_LIST = dataset_dict_to_list(DATASET_ITEMS_TEST)

len(DATASET_ITEMS_TRAIN_LIST), len(DATASET_ITEMS_TEST_LIST)


In [ ]:
class DigitDataset(Dataset):
    def __init__(self, *, items, r):
        self.r = r
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert("L")
        img = ImageOps.invert(img)
        img = img.resize((self.r, self.r))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(arr).unsqueeze(0)  # (1, R, R)
        return tensor, label


full_train_ds = DigitDataset(items=DATASET_ITEMS_TRAIN_LIST, r=RESOLUTION)
test_ds = DigitDataset(items=DATASET_ITEMS_TEST_LIST, r=RESOLUTION)
len(full_train_ds), len(test_ds)


## 4. Tunable Model

A small CNN whose width, depth, dropout and activation are chosen by Optuna.
The model intentionally omits a final `Softmax` layer because
`nn.CrossEntropyLoss` expects raw logits (per the PyTorch docs).


In [ ]:
ACTIVATIONS = {
    "relu": nn.ReLU,
    "gelu": nn.GELU,
    "leaky_relu": nn.LeakyReLU,
}


class TunableCNN(nn.Module):
    def __init__(
        self,
        *,
        r: int,
        conv_channels: tuple[int, ...] = (16, 32),
        hidden_dim: int = 128,
        dropout: float = 0.25,
        activation: str = "relu",
    ):
        super().__init__()
        act_cls = ACTIVATIONS[activation]

        layers: list[nn.Module] = []
        in_ch = 1
        for out_ch in conv_channels:
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                act_cls(),
                nn.MaxPool2d(2, 2),
            ]
            in_ch = out_ch

        self.features = nn.Sequential(*layers)

        reduced = r // (2 ** len(conv_channels))
        flat_dim = in_ch * reduced * reduced

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(flat_dim, hidden_dim),
            act_cls(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## 5. Training Logger

A `TrainingLog` accumulates everything needed for plotting:

- per-step training loss
- per-epoch train / val loss & accuracy
- per-epoch L2-norm, mean, std of every parameter tensor (weights & gradients)
- per-epoch learning rate
- wall-clock time per epoch


In [ ]:
@dataclass
class TrainingLog:
    step_loss: list[float] = field(default_factory=list)

    train_loss: list[float] = field(default_factory=list)
    train_acc: list[float] = field(default_factory=list)
    val_loss: list[float] = field(default_factory=list)
    val_acc: list[float] = field(default_factory=list)

    lr: list[float] = field(default_factory=list)
    epoch_time: list[float] = field(default_factory=list)

    # {param_name: [stat_per_epoch, ...]}
    weight_norm: dict[str, list[float]] = field(default_factory=lambda: defaultdict(list))
    weight_mean: dict[str, list[float]] = field(default_factory=lambda: defaultdict(list))
    weight_std: dict[str, list[float]] = field(default_factory=lambda: defaultdict(list))
    grad_norm: dict[str, list[float]] = field(default_factory=lambda: defaultdict(list))
    grad_mean: dict[str, list[float]] = field(default_factory=lambda: defaultdict(list))
    grad_std: dict[str, list[float]] = field(default_factory=lambda: defaultdict(list))


def _record_param_stats(log: TrainingLog, model: nn.Module) -> None:
    """Snapshot weights & gradients of each named parameter."""
    for name, p in model.named_parameters():
        with torch.no_grad():
            w = p.detach()
            log.weight_norm[name].append(w.norm().item())
            log.weight_mean[name].append(w.mean().item())
            log.weight_std[name].append(w.std().item() if w.numel() > 1 else 0.0)

            if p.grad is not None:
                g = p.grad.detach()
                log.grad_norm[name].append(g.norm().item())
                log.grad_mean[name].append(g.mean().item())
                log.grad_std[name].append(g.std().item() if g.numel() > 1 else 0.0)
            else:
                log.grad_norm[name].append(float("nan"))
                log.grad_mean[name].append(float("nan"))
                log.grad_std[name].append(float("nan"))


## 6. Training Loop

In [ ]:
@dataclass
class TrainOutcome:
    model: nn.Module
    log: TrainingLog
    best_val_acc: float
    best_val_loss: float


def evaluate(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += X.size(0)
    return total_loss / n, correct / n


def train_with_tracking(
    *,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader | None,
    epochs: int,
    lr: float,
    weight_decay: float = 0.0,
    optimizer_name: str = "adam",
    echo: bool = True,
) -> TrainOutcome:
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay
        )
    else:
        raise ValueError(optimizer_name)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    log = TrainingLog()
    best_acc, best_loss = -math.inf, math.inf

    for epoch in range(epochs):
        epoch_start = time.perf_counter()
        model.train()

        running_loss, correct, n = 0.0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

            log.step_loss.append(loss.item())
            running_loss += loss.item() * X.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += X.size(0)

        # snapshot params + last-step grads (still populated before next zero_grad)
        _record_param_stats(log, model)

        train_loss = running_loss / n
        train_acc = correct / n
        log.train_loss.append(train_loss)
        log.train_acc.append(train_acc)
        log.lr.append(optimizer.param_groups[0]["lr"])

        if val_loader is not None:
            v_loss, v_acc = evaluate(model, val_loader, loss_fn)
            log.val_loss.append(v_loss)
            log.val_acc.append(v_acc)
            best_acc = max(best_acc, v_acc)
            best_loss = min(best_loss, v_loss)

        scheduler.step()
        log.epoch_time.append(time.perf_counter() - epoch_start)

        if echo:
            v_str = (
                f"  val_loss={log.val_loss[-1]:.4f}  val_acc={log.val_acc[-1]:.2%}"
                if val_loader is not None
                else ""
            )
            print(
                f"Epoch {epoch + 1:2d}  loss={train_loss:.4f}  acc={train_acc:.2%}{v_str}"
            )

    return TrainOutcome(
        model=model,
        log=log,
        best_val_acc=best_acc if val_loader is not None else train_acc,
        best_val_loss=best_loss if val_loader is not None else train_loss,
    )


## 7. Hyperparameter Tuning — Optuna + K-Fold (cv = 5)

Each Optuna trial samples a hyperparameter set, then evaluates it via 5-fold
cross-validation on the training partition. The mean validation accuracy
across folds is the objective Optuna maximises.


In [ ]:
def build_loaders(train_idx, val_idx, batch_size: int):
    train_subset = Subset(full_train_ds, train_idx)
    val_subset = Subset(full_train_ds, val_idx)

    train_loader = DataLoader(
        train_subset, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=(device != "cpu"),
    )
    val_loader = DataLoader(
        val_subset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=(device != "cpu"),
    )
    return train_loader, val_loader


def sample_params(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "lr": trial.suggest_float("lr", 1e-4, 5e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512]),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 256]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "activation": trial.suggest_categorical("activation", list(ACTIVATIONS)),
        "optimizer_name": trial.suggest_categorical("optimizer_name", ["adam", "adamw", "sgd"]),
        "n_conv_layers": trial.suggest_int("n_conv_layers", 1, 3),
        "base_channels": trial.suggest_categorical("base_channels", [8, 16, 32]),
    }


def make_model(p: dict[str, Any]) -> TunableCNN:
    channels = tuple(p["base_channels"] * (2 ** i) for i in range(p["n_conv_layers"]))
    return TunableCNN(
        r=RESOLUTION,
        conv_channels=channels,
        hidden_dim=p["hidden_dim"],
        dropout=p["dropout"],
        activation=p["activation"],
    )


# Storage so plots can later inspect every trial's per-fold log.
TRIAL_LOGS: dict[int, list[TrainingLog]] = {}


def objective(trial: optuna.Trial) -> float:
    p = sample_params(trial)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    all_indices = np.arange(len(full_train_ds))

    fold_accs: list[float] = []
    fold_logs: list[TrainingLog] = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
        train_loader, val_loader = build_loaders(train_idx, val_idx, p["batch_size"])
        model = make_model(p)
        outcome = train_with_tracking(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=TUNE_EPOCHS,
            lr=p["lr"],
            weight_decay=p["weight_decay"],
            optimizer_name=p["optimizer_name"],
            echo=False,
        )
        fold_accs.append(outcome.best_val_acc)
        fold_logs.append(outcome.log)

        # Pruning: report intermediate fold result
        trial.report(float(np.mean(fold_accs)), step=fold)
        if trial.should_prune():
            TRIAL_LOGS[trial.number] = fold_logs
            raise optuna.TrialPruned()

    TRIAL_LOGS[trial.number] = fold_logs
    return float(np.mean(fold_accs))


In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
    study_name="ml-digits-cnn",
)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print("Best value:", study.best_value)
print("Best params:", study.best_params)


## 8. Final Retrain on Best Params

In [ ]:
best = study.best_params
best_model_params = {
    "hidden_dim": best["hidden_dim"],
    "dropout": best["dropout"],
    "activation": best["activation"],
    "n_conv_layers": best["n_conv_layers"],
    "base_channels": best["base_channels"],
}

# 80/20 train/val split of the existing 70% training partition
g = torch.Generator().manual_seed(SEED)
n_total = len(full_train_ds)
n_val = int(0.2 * n_total)
n_train = n_total - n_val
train_split, val_split = torch.utils.data.random_split(
    full_train_ds, [n_train, n_val], generator=g
)

final_train_loader = DataLoader(
    train_split, batch_size=best["batch_size"], shuffle=True,
    pin_memory=(device != "cpu"),
)
final_val_loader = DataLoader(
    val_split, batch_size=best["batch_size"], shuffle=False,
    pin_memory=(device != "cpu"),
)
test_loader = DataLoader(
    test_ds, batch_size=best["batch_size"], shuffle=False,
    pin_memory=(device != "cpu"),
)

final_outcome = train_with_tracking(
    model=make_model(best),
    train_loader=final_train_loader,
    val_loader=final_val_loader,
    epochs=FINAL_EPOCHS,
    lr=best["lr"],
    weight_decay=best["weight_decay"],
    optimizer_name=best["optimizer_name"],
    echo=True,
)

test_loss, test_acc = evaluate(final_outcome.model, test_loader, nn.CrossEntropyLoss())
print(f"\nTest loss = {test_loss:.4f}   test acc = {test_acc:.2%}")


## 9. Plots — Training Dynamics

In [ ]:
log = final_outcome.log

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log.train_loss, label="train")
axes[0].plot(log.val_loss, label="val")
axes[0].set_title("Loss per epoch")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()

axes[1].plot(log.train_acc, label="train")
axes[1].plot(log.val_acc, label="val")
axes[1].set_title("Accuracy per epoch")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(log.step_loss, alpha=0.4, label="step loss")
window = max(1, len(log.step_loss) // 100)
if window > 1:
    smooth = np.convolve(log.step_loss, np.ones(window) / window, mode="valid")
    plt.plot(range(window - 1, len(log.step_loss)), smooth, color="red", label=f"rolling mean (w={window})")
plt.title("Per-step training loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(log.lr, marker="o")
plt.title("Learning rate schedule (CosineAnnealingLR)")
plt.xlabel("epoch"); plt.ylabel("lr")
plt.tight_layout(); plt.show()


### Weight statistics per layer

In [ ]:
def plot_layer_stats(stats: dict, title: str, ylabel: str):
    fig, ax = plt.subplots(figsize=(12, 5))
    for name, values in stats.items():
        ax.plot(values, label=name)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel(ylabel)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


plot_layer_stats(log.weight_norm, "L2 norm of weights per layer", "||W||")
plot_layer_stats(log.weight_mean, "Mean of weights per layer", "mean(W)")
plot_layer_stats(log.weight_std, "Std of weights per layer", "std(W)")


### Gradient statistics per layer (last batch of each epoch)

In [ ]:
plot_layer_stats(log.grad_norm, "L2 norm of gradients per layer", "||grad||")
plot_layer_stats(log.grad_mean, "Mean of gradients per layer", "mean(grad)")
plot_layer_stats(log.grad_std, "Std of gradients per layer", "std(grad)")


## 10. Plots — Optuna

In [ ]:
ovm.plot_optimization_history(study); plt.tight_layout(); plt.show()
ovm.plot_param_importances(study); plt.tight_layout(); plt.show()
ovm.plot_parallel_coordinate(study); plt.tight_layout(); plt.show()
ovm.plot_slice(study); plt.tight_layout(); plt.show()


## 11. Confusion Matrix on Test Set

In [ ]:
final_outcome.model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        all_preds.append(final_outcome.model(X).argmax(1).cpu().numpy())
        all_labels.append(y.numpy())
preds = np.concatenate(all_preds); labels = np.concatenate(all_labels)

cm = np.zeros((10, 10), dtype=int)
for t, p in zip(labels, preds):
    cm[t, p] += 1

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("predicted"); plt.ylabel("true")
plt.title(f"Confusion matrix (acc = {(preds == labels).mean():.2%})")
plt.tight_layout(); plt.show()


## 12. Summary

- All hyperparameters were chosen by Optuna using 5-fold cross-validation.
- `TrainingLog` captured per-step loss, per-epoch train/val metrics, the LR
  schedule, and per-layer weight & gradient statistics — every quantity above
  is available as a Python list/dict for further analysis.
- `TRIAL_LOGS[trial_number]` keeps the per-fold logs of every Optuna trial
  in case you want to compare runs.
